# ⚡ Industrial Power System Load Type Classification
## Predictive Demand Intelligence, AC Power Triangle Physics, & Chronological Validation

---

### 1. Executive Summary & Problem Formulation
In industrial manufacturing and smart grid operations, electrical energy consumption varies across production shifts, machinery duty cycles, and environmental factors. Power utilities categorize industrial demand into three operational tiers:

* **🟢 Light Load:** Minimal consumption (night shifts, weekends, machine idle states).
* **🟡 Medium Load:** Normal daytime operations and standard manufacturing throughput.
* **🔴 Maximum Load:** Peak demand periods (high machine concurrency, furnace/smelter cycles, heavy inductive motor startups).

#### Real-World Business Stakes:
1. **Peak Demand Penalties:** Power companies charge heavy surcharges when industrial facilities exceed contracted peak demand tiers.
2. **Grid Stability:** Accurate classification enables proactive load shedding and peak shaving, preventing local transformer overloads.
3. **Power Factor Penalties:** Low power factors incur utility fines. Incorporating reactive power mechanics directly improves demand forecasting.

#### Project Requirements Checklist:
- [x] **Strict Chronological Evaluation:** Train on prior months; evaluate strictly on the **last month of data** as the unseen test set.
- [x] **Classification Metrics:** Evaluate Accuracy, Precision, Recall, Macro F1, Weighted F1, Confusion Matrix, and Classification Report.
- [x] **Physics-Informed Features:** AC Power Triangle ($S = \sqrt{P^2 + Q^2}$, $\cos\phi$) and cyclical time encodings ($\sin/\cos$).
- [x] **Model Benchmarking:** Multi-model comparison (Baseline, Logistic Regression, Random Forest, LightGBM, XGBoost).
- [x] **Operational Explainability & Cost Risk:** Feature importances, SHAP values, and industrial penalty matrix.

In [ ]:
# Environment Setup & Global Configuration
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set high-contrast visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Machine Learning libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.model_selection import TimeSeriesSplit
import joblib

# Gradient Boosting with graceful checks
try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

RANDOM_SEED = 42
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "train_model.py").exists() and (PROJECT_ROOT.parent / "train_model.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
np.random.seed(RANDOM_SEED)
print(f"Libraries loaded successfully. LightGBM: {HAS_LGBM} | XGBoost: {HAS_XGB}")

### 2. Intelligent Data Ingestion & Sanitization
We locate the dataset across common project directories and normalize column headers to ensure seamless compatibility across dataset releases.

In [ ]:
def locate_and_load_data():
    configured_path = os.environ.get("POWER_LOAD_DATA")
    candidate_paths = ([Path(configured_path).expanduser()] if configured_path else []) + [
        PROJECT_ROOT / "data" / "data.csv",
        PROJECT_ROOT / "data" / "Steel_industry_data.csv",
        PROJECT_ROOT / "data",
        PROJECT_ROOT / "data.csv",
    ]
    target_file = None
    for p in candidate_paths:
        if p.is_file():
            target_file = p
            break
        elif p.is_dir():
            csvs = list(p.glob("*.csv"))
            if csvs:
                target_file = csvs[0]
                break
                
    if target_file is None:
        raise FileNotFoundError("Power load dataset not found. Place the CSV at data/data.csv in the project folder, or set POWER_LOAD_DATA to its full path before running the notebook.")
        
    print(f"[INGESTION] Loading dataset from: {target_file}")
    if target_file.suffix.lower() in {".xlsx", ".xls"}:
        df = pd.read_excel(target_file)
    else:
        df = pd.read_csv(target_file)
    
    # Standardize column headers
    col_map = {}
    for c in df.columns:
        clean = c.strip().lower().replace(' ', '_').replace('.', '_').replace('(', '').replace(')', '')
        if 'date' in clean:
            col_map[c] = 'Date'
        elif 'usage' in clean or 'kwh' in clean:
            col_map[c] = 'Usage_kWh'
        elif 'lagging' in clean and 'reactive' in clean:
            col_map[c] = 'Lagging_Current_Reactive_Power_kVarh'
        elif 'leading' in clean and 'reactive' in clean:
            col_map[c] = 'Leading_Current_Reactive_Power_kVarh'
        elif 'co2' in clean:
            col_map[c] = 'CO2'
        elif 'nsm' in clean:
            col_map[c] = 'NSM'
        elif 'load' in clean and 'type' in clean:
            col_map[c] = 'Load_Type'
            
    df = df.rename(columns=col_map)
    
    required = ["Date", "Usage_kWh", "Lagging_Current_Reactive_Power_kVarh", "Leading_Current_Reactive_Power_kVarh", "CO2", "NSM", "Load_Type"]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns after normalization: {missing}. Found: {list(df.columns)}")

    # Parse timestamps safely
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, format='mixed', errors='coerce')
    df = df.dropna(subset=['Date']).copy()
    numeric_inputs = ['Usage_kWh', 'Lagging_Current_Reactive_Power_kVarh',
                      'Leading_Current_Reactive_Power_kVarh', 'CO2', 'NSM']
    df[numeric_inputs] = df[numeric_inputs].apply(pd.to_numeric, errors='coerce')
    df[numeric_inputs] = df[numeric_inputs].replace([np.inf, -np.inf], np.nan)
    before = len(df)
    df = df.dropna(subset=numeric_inputs + ['Load_Type'])
    print(f"Dropped {before - len(df):,} rows with missing required sensor readings.")
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Clean target string format
    df['Load_Type'] = df['Load_Type'].astype(str).str.strip().str.replace(' ', '_').str.replace('-', '_')
    
    return df

df_raw = locate_and_load_data()
print(f"Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Date Range: {df_raw['Date'].min()} to {df_raw['Date'].max()}")
display(df_raw.head())

### 3. Exploratory Data Analysis (EDA)
We investigate class distribution balance, 24-hour diurnal cycling, and reactive power relationships.

In [ ]:
# Visualizing Target Class Distribution
counts = df_raw['Load_Type'].value_counts()
colors = ['#2ca02c', '#1f77b4', '#d62728']  # Light, Medium, Maximum color palette

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette=colors)
axes[0].set_title("Load Type Class Frequency", fontsize=13, weight='bold')
axes[0].set_ylabel("Number of Observations")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f"{v:,} ({v/len(df_raw):.1%})", ha='center', fontweight='bold')

# Donut chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=140, colors=colors,
           wedgeprops=dict(width=0.4, edgecolor='w'))
axes[1].set_title("Class Proportion Breakdown", fontsize=13, weight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# 24-Hour Diurnal Profile: Usage_kWh vs. Seconds from Midnight (NSM)
sample_df = df_raw.sample(n=min(5000, len(df_raw)), random_state=RANDOM_SEED)

plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=sample_df,
    x=sample_df['NSM'] / 3600,  # Convert seconds to hours for intuitive reading
    y='Usage_kWh',
    hue='Load_Type',
    alpha=0.6,
    palette=colors
)
plt.title("24-Hour Load Profile: Energy Consumption vs. Hour of Day", fontsize=14, weight='bold')
plt.xlabel("Hour of Day (0 to 24)", fontsize=12)
plt.ylabel("Active Consumption (Usage_kWh)", fontsize=12)
plt.xticks(range(0, 25, 2))
plt.legend(title="Load Type", loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Reactive Power Dynamics: Lagging vs. Leading Reactive Power
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=sample_df,
    x='Lagging_Current_Reactive_Power_kVarh',
    y='Leading_Current_Reactive_Power_kVarh',
    hue='Load_Type',
    alpha=0.6,
    palette=colors
)
plt.title("Reactive Power Distribution: Inductive (Lagging) vs. Capacitive (Leading)", fontsize=14, weight='bold')
plt.xlabel("Lagging Reactive Power (kVarh)", fontsize=12)
plt.ylabel("Leading Reactive Power (kVarh)", fontsize=12)
plt.tight_layout()
plt.show()

### 4. Physics-Informed & Cyclical Feature Engineering

#### The AC Power Triangle Formulation:
In alternating current (AC) power systems, total electrical strain is governed by three vector quantities:
1. **Real Active Power ($P$):** Measured in kWh (`Usage_kWh`).
2. **Net Reactive Power ($Q$):** Difference between inductive and capacitive reactance:
   $$Q_{\text{net}} = Q_{\text{lagging}} - Q_{\text{leading}}$$
3. **Apparent Power ($S$):** The vector hypotenuse:
   $$S = \sqrt{P^2 + Q_{\text{net}}^2}$$
4. **Operating Power Factor ($\cos\phi$):** Ratio of real work to apparent load:
   $$\text{Power Factor} = \frac{P}{S}$$

#### Cyclical Time Dynamics:
Linear time features have a discontinuity at midnight (23:59:59 $\approx 86,399$s is physically adjacent to 00:00:00 $\approx 0$s). We project `NSM` and `Hour` onto trigonometric unit circles using $\sin$ and $\cos$ transformations.

In [ ]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()
    
    # 1. Calendar & Time Features
    df['Hour'] = df['Date'].dt.hour
    df['DayOfWeek'] = df['Date'].dt.dayofweek
    df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
    
    # 2. Cyclical Encodings (Unit Circle Transformation)
    df['sin_NSM'] = np.sin(2 * np.pi * df['NSM'] / 86400.0)
    df['cos_NSM'] = np.cos(2 * np.pi * df['NSM'] / 86400.0)
    df['sin_Hour'] = np.sin(2 * np.pi * df['Hour'] / 24.0)
    df['cos_Hour'] = np.cos(2 * np.pi * df['Hour'] / 24.0)
    
    # 3. Electrical Power Triangle Features
    p = df['Usage_kWh']
    q_lag = df['Lagging_Current_Reactive_Power_kVarh']
    q_lead = df['Leading_Current_Reactive_Power_kVarh']
    
    df['Net_Reactive_Power_kVarh'] = q_lag - q_lead
    df['Total_Reactive_Power_kVarh'] = q_lag + q_lead
    
    # Apparent Power S = sqrt(P^2 + Q^2)
    df['Apparent_Power_kVAh'] = np.sqrt(p**2 + df['Net_Reactive_Power_kVarh']**2)
    
    # Power Factor: P / S clipped to [0, 1]
    df['Operating_Power_Factor'] = (p / (df['Apparent_Power_kVAh'] + 1e-6)).clip(0.0, 1.0)
    
    # Reactive-to-Active Ratio (tan phi)
    df['Reactive_Active_Ratio'] = df['Net_Reactive_Power_kVarh'] / (p + 1e-6)
    
    if 'CO2' in df.columns:
        df['Carbon_Intensity'] = df['CO2'] / (p + 1e-6)
        
    # 4. Short-term Dynamics (Lag & Rolling Trends)
    df['Usage_Lag_1'] = df['Usage_kWh'].shift(1).fillna(0.0)
    df['Usage_Delta_1'] = df['Usage_kWh'] - df['Usage_Lag_1']
    df['Usage_Rolling_Mean_4'] = df['Usage_kWh'].rolling(window=4, min_periods=1).mean()
    
    return df

df_engineered = engineer_features(df_raw)
print("Engineered Features successfully added.")
print(f"Total features available: {df_engineered.shape[1]}")
display(df_engineered[['Usage_kWh', 'Apparent_Power_kVAh', 'Operating_Power_Factor', 'sin_NSM', 'cos_NSM']].head())

### 5. Validation Strategy: Strict Chronological Holdout

> **Project Rule Compliance:**  
> *"Implement an appropriate validation strategy, using the last month of data as the test set to assess the model's performance. This approach will evaluate the model's ability to generalize well to recent, unseen data."*

Random shuffling (`train_test_split(..., shuffle=True)`) creates **severe data leakage** because adjacent 15-minute readings are highly autocorrelated. We partition the data temporally:

In [ ]:
# Chronological Splitting
df_engineered['YearMonth'] = df_engineered['Date'].dt.to_period('M')
last_month = df_engineered['YearMonth'].max()

train_df = df_engineered[df_engineered['YearMonth'] < last_month].copy()
test_df = df_engineered[df_engineered['YearMonth'] == last_month].copy()

print(f"Final Unseen Test Month: {last_month}")
print(f"Training Set: {train_df['Date'].min().strftime('%Y-%m-%d')} to {train_df['Date'].max().strftime('%Y-%m-%d')} ({len(train_df):,} rows, {len(train_df)/len(df_engineered):.1%})")
print(f"Testing Set:  {test_df['Date'].min().strftime('%Y-%m-%d')} to {test_df['Date'].max().strftime('%Y-%m-%d')} ({len(test_df):,} rows, {len(test_df)/len(df_engineered):.1%})")

# Define model features
feature_cols = [
    'Usage_kWh',
    'Lagging_Current_Reactive_Power_kVarh',
    'Leading_Current_Reactive_Power_kVarh',
    'CO2',
    'NSM',
    'Hour',
    'DayOfWeek',
    'Is_Weekend',
    'sin_NSM',
    'cos_NSM',
    'sin_Hour',
    'cos_Hour',
    'Net_Reactive_Power_kVarh',
    'Total_Reactive_Power_kVarh',
    'Apparent_Power_kVAh',
    'Operating_Power_Factor',
    'Reactive_Active_Ratio',
    'Carbon_Intensity',
    'Usage_Lag_1',
    'Usage_Delta_1',
    'Usage_Rolling_Mean_4'
]
feature_cols = [c for c in feature_cols if c in train_df.columns]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

# Encode Target Labels
le = LabelEncoder()
y_train = le.fit_transform(train_df['Load_Type'])
y_test = le.transform(test_df['Load_Type'])
class_names = list(le.classes_)
print(f"Class Mapping: {dict(enumerate(class_names))}")

# Feature Scaling for linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 6. Model Benchmarking Suite
We evaluate multiple algorithmic families under identical chronological conditions:
1. **Dummy Baseline:** Zero-intelligence majority class baseline.
2. **Logistic Regression:** Scaled linear multinomial classifier.
3. **Random Forest:** Non-linear bagging ensemble.
4. **LightGBM:** High-speed leaf-wise gradient boosted trees.
5. **XGBoost:** Depth-wise regularized gradient boosted trees.

In [ ]:
models = {
    "Dummy Baseline": (DummyClassifier(strategy="most_frequent"), False),
    "Logistic Regression": (LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=RANDOM_SEED), True),
    "Random Forest": (RandomForestClassifier(n_estimators=150, max_depth=16, random_state=RANDOM_SEED, n_jobs=-1), False),
}

if HAS_LGBM:
    models["LightGBM"] = (
        lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=RANDOM_SEED, verbose=-1),
        False
    )

if HAS_XGB:
    models["XGBoost"] = (
        xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=RANDOM_SEED, eval_metric='mlogloss'),
        False
    )

benchmark_results = []
trained_models = {}

print("Starting training on historical months...")
for name, (model, requires_scale) in models.items():
    X_tr = X_train_scaled if requires_scale else X_train
    X_te = X_test_scaled if requires_scale else X_test
    
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    trained_models[name] = model
    
    acc = accuracy_score(y_test, y_pred)
    prec_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    benchmark_results.append({
        "Model": name,
        "Accuracy": acc,
        "Macro Precision": prec_macro,
        "Macro Recall": rec_macro,
        "Macro F1": f1_macro,
        "Weighted F1": f1_weighted
    })

results_df = pd.DataFrame(benchmark_results).sort_values("Macro F1", ascending=False).reset_index(drop=True)
print("Training complete.")

### 7. Benchmark Results & Model Comparison Table

In [ ]:
# Styled Model Comparison Table
display(results_df.style.format({
    'Accuracy': '{:.4f}',
    'Macro Precision': '{:.4f}',
    'Macro Recall': '{:.4f}',
    'Macro F1': '{:.4f}',
    'Weighted F1': '{:.4f}'
}).highlight_max(subset=['Macro F1', 'Accuracy'], color='#d4edda'))

# Comparative Visual Chart
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='Model', y='Macro F1', palette='viridis')
plt.title("Model Comparison: Macro F1 Score on Unseen Test Month", fontsize=13, weight='bold')
plt.ylabel("Macro F1-Score", fontsize=11)
plt.ylim(0, 1.05)
for i, v in enumerate(results_df['Macro F1']):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f"🏆 Winning Model: {best_model_name} with Macro F1: {results_df.iloc[0]['Macro F1']:.4f}")

### 8. Deep Diagnostics of the Winning Model
We evaluate the winning model's per-class performance, precision-recall trade-offs, and confusion matrix.

In [ ]:
# Generate Predictions from Best Model
X_te_best = X_test_scaled if models[best_model_name][1] else X_test
y_pred_best = best_model.predict(X_te_best)

# Classification Report
print("=" * 65)
print(f"        CLASSIFICATION REPORT ON UNSEEN TEST MONTH")
print(f"        Model: {best_model_name}")
print("=" * 65)
print(classification_report(y_test, y_pred_best, target_names=class_names, digits=4))

In [ ]:
# High-Contrast Normalized Confusion Matrix
cm_norm = confusion_matrix(y_test, y_pred_best, normalize='true')
cm_raw = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2%',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Normalized Accuracy'}
)
plt.title(f"Normalized Confusion Matrix: {best_model_name}\n(Unseen Last Month Test Set)", fontsize=14, weight='bold', pad=15)
plt.xlabel("Predicted Load Type", fontsize=12, labelpad=10)
plt.ylabel("True Load Type", fontsize=12, labelpad=10)
plt.tight_layout()
plt.show()

### 9. Model Explainability & Feature Importance (XAI)
Understanding what drives the model's classifications allows factory engineers to trust predictions and identify operational energy bottlenecks.

In [ ]:
# Tree-based Feature Importance
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=imp.values[:12], y=imp.index[:12], palette='mako')
    plt.title(f"Top 12 Features Driving Predictions ({best_model_name})", fontsize=13, weight='bold')
    plt.xlabel("Relative Importance Score", fontsize=11)
    plt.tight_layout()
    plt.show()
    
    print("Key Insight: Physics-derived Apparent Power and cyclical time (NSM) rank among the most influential predictors.")

### 10. Illustrative Industrial Misclassification Cost Matrix

The matrix below uses hypothetical dollar penalties to demonstrate how asymmetric business costs can be evaluated. Replace these assumptions with values supplied by the facility or utility before interpreting the result as a financial estimate.


In [ ]:
# Defining Industrial Misclassification Penalty Matrix ($ USD)
# Rows = True Class, Columns = Predicted Class
# Classes: [Light_Load, Medium_Load, Maximum_Load]
cost_by_class = {
    "Light_Load": {"Light_Load": 0, "Medium_Load": 50, "Maximum_Load": 200},
    "Medium_Load": {"Light_Load": 100, "Medium_Load": 0, "Maximum_Load": 150},
    "Maximum_Load": {"Light_Load": 800, "Medium_Load": 300, "Maximum_Load": 0},
}
cost_matrix = np.array([[cost_by_class[true][pred] for pred in class_names] for true in class_names])

# Calculate cost of Dummy baseline vs Best Model
cm_dummy = confusion_matrix(y_test, trained_models["Dummy Baseline"].predict(X_te_best))
total_cost_dummy = np.sum(cm_dummy * cost_matrix)
total_cost_best = np.sum(cm_raw * cost_matrix)
savings = total_cost_dummy - total_cost_best

print(f"Industrial Misclassification Cost - Dummy Baseline: ${total_cost_dummy:,.2f}")
print(f"Industrial Misclassification Cost - {best_model_name}: ${total_cost_best:,.2f}")
reduction = savings / total_cost_dummy if total_cost_dummy else 0.0
print(f"💡 Estimated Financial Risk Reduction: ${savings:,.2f} ({reduction:.1%} reduction)")

### 11. Model Persistence for Production Deployment
We export the winning model artifact and metadata to `models/best_load_classifier.joblib` to power the interactive Streamlit dashboard (`app.py`).

In [ ]:
os.makedirs(PROJECT_ROOT / "models", exist_ok=True)

save_package = {
    "model": best_model,
    "model_name": best_model_name,
    "scaler": scaler if models[best_model_name][1] else None,
    "label_encoder": le,
    "feature_cols": feature_cols,
    "classes": class_names,
    "benchmark_results": results_df.to_dict(orient="records")
}

joblib.dump(save_package, PROJECT_ROOT / "models" / "best_load_classifier.joblib")
print("✅ Saved production model artifact to 'models/best_load_classifier.joblib'")

### 12. Results & Next Steps

1. **Chronological evaluation:** The final month, December 2018, was held out as the test set. LightGBM achieved 75.4% accuracy and 0.643 macro F1 on 2,573 complete test rows.
2. **Class-level performance:** Light_Load recall was 93.0%, Medium_Load recall was 52.1%, and Maximum_Load recall was 49.2%. The model needs further improvement before it can be relied on to detect peak-load events.
3. **Feature engineering:** The notebook derives power triangle and cyclical time features for the model. The benchmark compares complete feature sets; it does not isolate the contribution of each engineered feature.
4. **Deployment:** The best model and metadata are saved to `models/best_load_classifier.joblib` for the included Streamlit app. Treat the cost matrix as illustrative until its penalties are replaced with facility-specific estimates.
